# Week 2: Data Structures & File I/O for Biology

**Goal:** Learn to read, parse, and write biology data files — the bread and butter of computational biology.

By the end of this notebook, you will:
- Manipulate DNA strings (complement, reverse complement, transcription)
- Read and parse FASTA files
- Read and write CSV files with gene expression data
- Handle errors gracefully when files have missing data
- Use list comprehensions for concise data filtering

**Time:** ~2-3 hours | **Difficulty:** Beginner

---

## 1. String Methods for DNA Sequences

Strings are your primary data type for sequences. Python has powerful built-in methods for manipulating them.

In [ ]:
dna = "atgcGATCgatcAATTCCGG"

# Case conversion — critical for consistent analysis
print(f"Original:   {dna}")
print(f"Uppercase:  {dna.upper()}")
print(f"Lowercase:  {dna.lower()}")

# Always normalize first!
dna = dna.upper()
print(f"\nNormalized: {dna}")
print(f"Length:     {len(dna)} bases")

In [ ]:
# Counting bases
print(f"A count: {dna.count('A')}")
print(f"T count: {dna.count('T')}")
print(f"G count: {dna.count('G')}")
print(f"C count: {dna.count('C')}")

# Finding motifs
motif = "AATTC"  # EcoRI recognition site
position = dna.find(motif)
if position != -1:
    print(f"\nEcoRI site (AATTC) found at position {position}")
else:
    print(f"\nEcoRI site not found")

In [ ]:
# Slicing — extracting parts of a sequence
# Think of it as cutting a gel band at specific positions

print(f"Full sequence:  {dna}")
print(f"First 5 bases:  {dna[:5]}")
print(f"Last 5 bases:   {dna[-5:]}")
print(f"Bases 5-10:     {dna[5:10]}")
print(f"Every 3rd base: {dna[::3]}")
print(f"Reversed:       {dna[::-1]}")

In [ ]:
# Building a complete sequence toolkit

def complement(dna):
    """Return the complement of a DNA sequence."""
    comp_map = str.maketrans('ATGC', 'TACG')
    return dna.upper().translate(comp_map)

def reverse_complement(dna):
    """Return the reverse complement of a DNA sequence."""
    return complement(dna)[::-1]

def transcribe(dna):
    """Transcribe DNA to mRNA (coding strand → mRNA)."""
    return dna.upper().replace('T', 'U')

def gc_content(dna):
    """Calculate GC content as a percentage."""
    dna = dna.upper()
    if len(dna) == 0:
        return 0.0
    return (dna.count('G') + dna.count('C')) / len(dna) * 100


# Demo
seq = "ATGCGATCGATCG"
print(f"DNA:                 5'-{seq}-3'")
print(f"Complement:          3'-{complement(seq)}-5'")
print(f"Reverse complement:  5'-{reverse_complement(seq)}-3'")
print(f"mRNA:                5'-{transcribe(seq)}-3'")
print(f"GC content:          {gc_content(seq):.1f}%")

### Try it yourself!
Find all occurrences of the motif "ATG" (start codon) in the sequence below.

In [ ]:
# YOUR TURN: Find all start codons (ATG) in this sequence
sequence = "CGATGCCCATGATGCCCATGAAA"

# Hint: use a while loop with str.find(motif, start_position)
positions = []
start = 0
while True:
    pos = sequence.find("ATG", start)
    if pos == -1:
        break
    positions.append(pos)
    start = pos + 1  # move past this occurrence

print(f"Sequence: {sequence}")
print(f"ATG found at positions: {positions}")
print(f"Total start codons: {len(positions)}")

---

## 2. Reading FASTA Files

FASTA is the most common sequence file format in biology. Each entry has:
- A header line starting with `>`
- One or more lines of sequence

Let's first create a sample FASTA file, then learn to parse it.

In [ ]:
# Create a sample FASTA file to work with
fasta_content = """>BRCA1_human Breast cancer type 1 susceptibility protein
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAA
AATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGA
CCACATATTTTGCAAATTTTGCATGCTGAAACTTCTCAACCAGAAGAAAGGGCCTTCAC
>TP53_human Cellular tumor antigen p53
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP
DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYPQGLNGTVNLPGRNSFEV
RVCACPGRDRRTEEENLHKTTGIDSFLHPSVFVTPM
>EGFR_human Epidermal growth factor receptor
MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVV
LGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAV
>KRAS_human GTPase KRas
MTEYKLVVVGAVGVGKSALTIQLIQNHFVDEYDPTIEDYRKQVVIDGETCLLDILDTAG
QEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHQRQVHRDLREQKLISEEDL
>MYC_human Myc proto-oncogene protein
MDFFRVVENQQPATMPLNVSFTNRNYDLDYDSVQPYFYCDEEENFYQQQQQSELQPPAPS
EDIWKKFELLPTPPLSPSRRSGLCSPSYVAVTPFSLRGDNDGGGGSFSTADQLEMVTEL"""

# Write it to a file
with open("sample_sequences.fasta", "w") as f:
    f.write(fasta_content)

print("Created sample_sequences.fasta")
print(f"File size: {len(fasta_content)} characters")

In [ ]:
# Parse the FASTA file

def parse_fasta(filename):
    """Parse a FASTA file and return a dictionary of {name: sequence}.
    
    Parameters:
        filename (str): Path to the FASTA file
    
    Returns:
        dict: {sequence_name: sequence_string}
    """
    sequences = {}
    current_name = None
    current_seq = []
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()  # Remove whitespace and newlines
            
            if line.startswith('>'):
                # Save previous sequence if it exists
                if current_name is not None:
                    sequences[current_name] = ''.join(current_seq)
                
                # Start new sequence
                current_name = line[1:]  # Remove the '>'
                current_seq = []
            else:
                # This is a sequence line
                current_seq.append(line)
        
        # Don't forget the last sequence!
        if current_name is not None:
            sequences[current_name] = ''.join(current_seq)
    
    return sequences


# Parse our file
seqs = parse_fasta("sample_sequences.fasta")

print(f"Parsed {len(seqs)} sequences:\n")
for name, seq in seqs.items():
    print(f"  {name}")
    print(f"    Length: {len(seq)} residues")
    print(f"    First 30: {seq[:30]}...")
    print()

In [ ]:
# Analyze the parsed sequences

print(f"{'Name':<20} {'Length':>8} {'First 40 residues'}")
print("=" * 72)

lengths = []
for name, seq in seqs.items():
    short_name = name.split()[0]  # Just the gene ID
    lengths.append(len(seq))
    print(f"{short_name:<20} {len(seq):>8} {seq[:40]}...")

print("=" * 72)
print(f"{'Total sequences:':<20} {len(seqs):>8}")
print(f"{'Average length:':<20} {sum(lengths)/len(lengths):>8.1f}")
print(f"{'Shortest:':<20} {min(lengths):>8}")
print(f"{'Longest:':<20} {max(lengths):>8}")

---

## 3. Reading and Writing CSV Files

CSV (Comma-Separated Values) files are how most tabular biology data is shared — gene expression matrices, sample metadata, analysis results.

In [ ]:
# Create a sample gene expression CSV
csv_content = """gene_id,gene_name,sample_A,sample_B,sample_C,sample_D,sample_E
ENSG00000141510,TP53,5.2,3.8,7.1,4.5,6.3
ENSG00000146648,EGFR,12.4,15.2,8.9,14.1,11.7
ENSG00000012048,BRCA1,3.1,2.9,4.2,3.5,
ENSG00000133703,KRAS,8.7,9.2,7.5,8.1,8.9
ENSG00000136997,MYC,22.1,18.5,25.3,,19.8
ENSG00000171862,PTEN,4.8,5.1,4.3,4.9,5.2
ENSG00000121879,PIK3CA,6.7,7.2,5.9,6.4,7.1
ENSG00000157764,BRAF,3.2,3.5,2.8,3.1,3.4"""

with open("gene_expression.csv", "w") as f:
    f.write(csv_content)

print("Created gene_expression.csv")
print("Note: BRCA1 sample_E and MYC sample_D have MISSING values (intentional!)")

In [ ]:
# Method 1: Read CSV with Python's built-in csv module
import csv

with open("gene_expression.csv", 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # First row is the header
    
    print(f"Columns: {header}")
    print(f"\nData rows:")
    
    for row in reader:
        gene_name = row[1]
        values = row[2:]  # Expression values
        print(f"  {gene_name}: {values}")

In [ ]:
# Method 2: Read CSV with csv.DictReader (access columns by name)
import csv

with open("gene_expression.csv", 'r') as f:
    reader = csv.DictReader(f)
    
    print("Genes with expression > 10 in sample_A:\n")
    
    # Need to re-read since we consumed the reader above
    f.seek(0)  # Go back to start of file
    reader = csv.DictReader(f)
    
    for row in reader:
        try:
            val = float(row['sample_A'])
            if val > 10:
                print(f"  {row['gene_name']}: {val}")
        except ValueError:
            print(f"  {row['gene_name']}: missing or invalid data")

In [ ]:
# Writing results to a new CSV
import csv

# Calculate average expression per gene (handling missing values)
results = []

with open("gene_expression.csv", 'r') as f:
    reader = csv.DictReader(f)
    samples = ['sample_A', 'sample_B', 'sample_C', 'sample_D', 'sample_E']
    
    for row in reader:
        values = []
        missing = 0
        for s in samples:
            if row[s].strip():  # Not empty
                try:
                    values.append(float(row[s]))
                except ValueError:
                    missing += 1
            else:
                missing += 1
        
        avg = sum(values) / len(values) if values else 0
        results.append({
            'gene_name': row['gene_name'],
            'avg_expression': round(avg, 2),
            'min_expression': round(min(values), 2) if values else 'NA',
            'max_expression': round(max(values), 2) if values else 'NA',
            'missing_samples': missing,
        })

# Write summary to new CSV
with open("expression_summary.csv", 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['gene_name', 'avg_expression', 
                                           'min_expression', 'max_expression', 
                                           'missing_samples'])
    writer.writeheader()
    writer.writerows(results)

print("Expression Summary:")
print(f"{'Gene':<10} {'Avg':>8} {'Min':>8} {'Max':>8} {'Missing':>8}")
print("-" * 46)
for r in results:
    print(f"{r['gene_name']:<10} {r['avg_expression']:>8} {r['min_expression']:>8} {r['max_expression']:>8} {r['missing_samples']:>8}")

print("\nSaved to expression_summary.csv")

---

## 4. Error Handling — Dealing with Messy Data

Real biology data is messy: missing values, wrong formats, corrupted files. `try/except` blocks let your code handle problems gracefully instead of crashing.

In [ ]:
# Without error handling — this crashes!
messy_values = ["5.2", "3.8", "NA", "7.1", "", "4.5", "not_a_number"]

# This would crash: float("NA") raises ValueError
# total = sum(float(v) for v in messy_values)  # DON'T RUN THIS

# With error handling — works fine!
clean_values = []
problems = []

for i, v in enumerate(messy_values):
    try:
        clean_values.append(float(v))
    except ValueError:
        problems.append(f"Position {i}: '{v}' is not a number")

print(f"Clean values: {clean_values}")
print(f"Average: {sum(clean_values) / len(clean_values):.2f}")
print(f"\nProblems found ({len(problems)}):")
for p in problems:
    print(f"  {p}")

In [ ]:
# Robust file reading function

def safe_read_expression(filename):
    """Read a gene expression CSV, handling missing values and file errors."""
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            data = {}
            
            for row in reader:
                gene = row.get('gene_name', 'Unknown')
                values = []
                
                for key, val in row.items():
                    if key.startswith('sample_'):
                        try:
                            values.append(float(val))
                        except (ValueError, TypeError):
                            pass  # Skip missing/invalid values
                
                data[gene] = values
            
            return data
    
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return {}
    except Exception as e:
        print(f"Error reading file: {e}")
        return {}


# Test with our file
data = safe_read_expression("gene_expression.csv")
for gene, vals in data.items():
    avg = sum(vals) / len(vals) if vals else 0
    print(f"{gene:<10} {len(vals)} samples, avg = {avg:.2f}")

# Test with a non-existent file (won't crash!)
print()
data2 = safe_read_expression("nonexistent_file.csv")

---

## 5. List Comprehensions — Concise Data Filtering

List comprehensions let you filter and transform data in a single line. They're one of Python's most useful features for data analysis.

In [ ]:
# Gene expression data
expression = {
    "TP53": 5.2, "EGFR": 12.4, "BRCA1": 3.1, "KRAS": 8.7,
    "MYC": 22.1, "PTEN": 4.8, "PIK3CA": 6.7, "BRAF": 3.2
}

# The long way (for loop)
highly_expressed = []
for gene, value in expression.items():
    if value > 7.0:
        highly_expressed.append(gene)
print(f"For loop result:          {highly_expressed}")

# The short way (list comprehension) — same result!
highly_expressed = [gene for gene, value in expression.items() if value > 7.0]
print(f"List comprehension result: {highly_expressed}")

In [ ]:
# More biology list comprehension examples

dna_sequences = [
    "ATGCGATCGATCG",
    "GGCCGGCCGGCC",
    "AATTAATTAATT",
    "GCGCGCGCGCGC",
    "ATATATATATAT",
]

# Get GC content for all sequences
gc_values = [gc_content(seq) for seq in dna_sequences]
print(f"GC values: {[f'{v:.1f}%' for v in gc_values]}")

# Filter GC-rich sequences (>50%)
gc_rich = [seq for seq in dna_sequences if gc_content(seq) > 50]
print(f"\nGC-rich sequences: {gc_rich}")

# Get lengths of sequences longer than 12 bases
long_seqs = [(seq, len(seq)) for seq in dna_sequences if len(seq) > 12]
print(f"\nLong sequences: {long_seqs}")

# Transcribe all sequences at once
mrna_list = [transcribe(seq) for seq in dna_sequences]
print(f"\nmRNAs: {mrna_list}")

In [ ]:
# Dictionary comprehension — also very useful!

# Create a dictionary of gene → GC content
gene_seqs = {
    "BRCA1": "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTAC",
    "TP53":  "TCCCCCTTGCCGTCTGGGCTTCTTGCATTCTGGG",
    "EGFR":  "AACATCTCCGAAAGCCAACAAGGAAATCCTCGAT",
    "KRAS":  "ATGACTGAATATAAACTTGTGGTAGTTGGAGCTG",
}

gc_dict = {gene: round(gc_content(seq), 1) for gene, seq in gene_seqs.items()}
print("Gene GC content:")
for gene, gc in gc_dict.items():
    print(f"  {gene}: {gc}%")

---

## 6. MILESTONE: FASTA File Analyzer

Put everything together: parse a FASTA file, compute statistics, and write results to CSV.

In [ ]:
# Create a larger FASTA file with DNA sequences for analysis
dna_fasta = """>gene_001 Hypothetical protein A
ATGCGATCGATCGATCGAAATTCCGGCGATCGATCGATCGCGATCGATCG
GCTAGCTAGCTAGCTGATCGATCGATCGATCGATCG
>gene_002 Kinase domain protein
GGCCGGCCGGCCAATTCCGGCCGGCCGGCCAATTAATTCCGG
>gene_003 Transcription factor B
ATATATCGATCGCGCGATATATCGATCGATATATCGATCGCGCGATAT
CGATCGATATCGATCGATCGATCGCGCG
>gene_004 Membrane receptor C
GCGCGCGCGCGCATATATGCGCGCGCGCGCATAT
>gene_005 DNA repair enzyme
ATGCCCGAATTCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
CGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
AATTCCGGATCGATCG
>gene_006 Signal peptide D
ATGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGG
>gene_007 Ribosomal protein L1
GCGCAATATGCGCAATATGCGCGCAATATGCGCAATATGCGCAATATGCGCGC
>gene_008 Heat shock protein
ATGGCGATCGATCGCGCGATCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCGATCG
>gene_009 Cytochrome oxidase
ATGCCCAAAGGGCCCAAAGGGCCCAAAGGGCCCAAAGGGCCCAAAGGG
>gene_010 Unknown function
AATTAATTAATTCCGGCCGGAATTAATTAATTCCGGCCGG"""

with open("dna_sequences.fasta", "w") as f:
    f.write(dna_fasta)

print("Created dna_sequences.fasta with 10 sequences")

In [ ]:
# Complete FASTA analysis pipeline

import csv

# Step 1: Parse the FASTA file
sequences = parse_fasta("dna_sequences.fasta")

# Step 2: Analyze each sequence
results = []

print(f"{'Gene':<12} {'Length':>7} {'GC%':>7} {'A':>5} {'T':>5} {'G':>5} {'C':>5}  Classification")
print("=" * 70)

for name, seq in sequences.items():
    gene_id = name.split()[0]
    seq = seq.upper()
    length = len(seq)
    gc = gc_content(seq)
    a_pct = seq.count('A') / length * 100
    t_pct = seq.count('T') / length * 100
    g_pct = seq.count('G') / length * 100
    c_pct = seq.count('C') / length * 100
    
    if gc > 60:
        classification = "GC-rich"
    elif gc < 40:
        classification = "AT-rich"
    else:
        classification = "Balanced"
    
    print(f"{gene_id:<12} {length:>7} {gc:>6.1f}% {a_pct:>4.1f}% {t_pct:>4.1f}% {g_pct:>4.1f}% {c_pct:>4.1f}%  {classification}")
    
    results.append({
        'gene_id': gene_id,
        'description': ' '.join(name.split()[1:]),
        'length': length,
        'gc_content': round(gc, 2),
        'a_percent': round(a_pct, 2),
        't_percent': round(t_pct, 2),
        'g_percent': round(g_pct, 2),
        'c_percent': round(c_pct, 2),
        'classification': classification,
    })

# Step 3: Summary statistics
all_lengths = [r['length'] for r in results]
all_gc = [r['gc_content'] for r in results]

print("=" * 70)
print(f"\nSummary:")
print(f"  Total sequences: {len(results)}")
print(f"  Avg length:      {sum(all_lengths)/len(all_lengths):.1f} bp")
print(f"  Avg GC content:  {sum(all_gc)/len(all_gc):.1f}%")
print(f"  GC-rich genes:   {sum(1 for r in results if r['classification'] == 'GC-rich')}")
print(f"  AT-rich genes:   {sum(1 for r in results if r['classification'] == 'AT-rich')}")
print(f"  Balanced genes:  {sum(1 for r in results if r['classification'] == 'Balanced')}")

In [ ]:
# Step 4: Write results to CSV

output_file = "sequence_analysis_results.csv"

with open(output_file, 'w', newline='') as f:
    fieldnames = ['gene_id', 'description', 'length', 'gc_content', 
                  'a_percent', 't_percent', 'g_percent', 'c_percent', 'classification']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"Results saved to {output_file}")
print(f"\nYou can open this CSV in Excel or Google Sheets!")

# Verify by reading it back
print(f"\nVerification — first 3 lines of {output_file}:")
with open(output_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 4:
            print(f"  {line.strip()}")

---

## Week 2 Complete!

**What you learned:**
- String methods for DNA manipulation (complement, reverse complement, transcription)
- Parsing FASTA files — the foundation of bioinformatics
- Reading and writing CSV files with the `csv` module
- Error handling with `try/except` for messy real-world data
- List comprehensions for concise data filtering
- Built a complete FASTA → analysis → CSV pipeline

**Next week:** You'll learn NumPy and Pandas — the power tools that make Python a serious data analysis platform. You'll load real biology datasets into DataFrames and start doing the kind of analysis you'd normally do in Excel, but faster and reproducibly.

**Challenge:** Try downloading a real FASTA file from [NCBI](https://www.ncbi.nlm.nih.gov/) for a gene you work with, and run the analysis pipeline on it!